# ZoomNeXt — Camouflaged Object Detection (Colab v3 — Repaired)

**Project:** Camouflaged Object Detection using Deep Learning  
**Model:** RN50_ZoomNeXt (28.458M parameters)  
**Version:** v3 (Phase 3.0.2 — Environment Repair)  

> **What changed from v1?** v1 pinned `torch (Colab pre-installed)` which does NOT exist for Python 3.12 (Colab's default). v3 uses Colab's pre-installed torch (compatible with Python 3.12) and installs only the 6 packages actually needed for inference — no more, no less.

---

## Instructions
1. **Runtime → Change runtime type → GPU** (T4 is sufficient)
2. Run cells top to bottom
3. Total runtime: **~3 minutes** (v1 took ~10 min due to reinstalling torch)

## Key Design Decisions (Phase 3.0.2 Root Cause Analysis)

| Problem in v1 | Root Cause | Fix in v3 |
|---------------|-----------|----------|
| `torch (Colab pre-installed) not found` | No Python 3.12 wheel exists for torch 2.1.2 | Use Colab's pre-installed torch (2.x, already py3.12-compatible) |
| `numpy.dtype size changed` | albumentations pulled numpy 2.x; torch compiled against numpy 1.x | Force `numpy<2` AFTER all installs |
| `torchvision import failed` | torch/torchvision version mismatch from partial reinstall | Don't reinstall torch; use Colab's matched pair |
| `opencv requires numpy>=2` | opencv 4.8.1.78 has no py3.12 wheel; pip upgraded to 5.x which pulled numpy 2 | Use `opencv-python-headless` (unpinned, latest with py3.12 support) + force numpy<2 |
| `jax requires numpy>=2` | jax is pre-installed by Colab, not by our project | Ignore — jax is not used by ZoomNeXt; forcing numpy<2 disables jax but that's fine |

## Cell 1 — Install Minimal Dependencies (Inference Only)

**Philosophy:** install ONLY what `infer.py` actually imports. Verified by tracing the import chain:

| Package | Why needed | Source |
|---------|-----------|--------|
| torch | model forward pass | Colab pre-installed (do NOT reinstall) |
| torchvision | timm dependency | Colab pre-installed (do NOT reinstall) |
| numpy | arrays everywhere | Colab pre-installed; force <2 after |
| opencv-python-headless | image I/O (cv2) | Install (unpinned for py3.12 compatibility) |
| timm | ResNet50 backbone creation | Install |
| einops | tensor rearrange in MHSIU/RGPU | Install |
| gdown | weight download from Google Drive | Install |

**NOT installed** (only needed for training/full evaluation, NOT for inference):
- albumentations, mmengine, PyYAML, pysodmetrics, scipy, matplotlib, colorlog, tqdm, adjustText

In [ ]:
# === Cell 1: Install Minimal Dependencies (Inference Only) ===
# Modern torch (2.11+ with CUDA 12.8) supports numpy 2.x natively.
# DO NOT pin numpy<2 — that breaks the scientific stack on the latest Colab.
import torch, torchvision
print(f'torch: {torch.__version__} (Colab pre-installed)')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Install ONLY the missing inference packages (numpy stays at Colab default, 2.x).
!pip install -q timm einops gdown

# Suppress pip dependency resolver warnings (they are about unrelated Colab packages).
import warnings
warnings.filterwarnings('ignore')

# Verify the minimal set
import cv2, timm, einops, numpy
print(f'\nMinimal dependency set verified:')
print(f'  torch={torch.__version__}')
print(f'  torchvision={torchvision.__version__}')
print(f'  numpy={numpy.__version__}')
print(f'  cv2={cv2.__version__}')
print(f'  timm={timm.__version__}')
print(f'  einops={einops.__version__}')
print(f'  CUDA: {torch.cuda.is_available()}')
print(f'\nNote: pip dependency warnings about cupy, jax, rasterio, shap, etc. are HARMLESS.')
print(f'These are unrelated Colab pre-installed packages. Our project does NOT use them.')

## Cell 2 — Clone Repository

In [ ]:
# === Cell 2: Clone Repository ===
import os

REPO_DIR = '/content/ZoomNeXt'

if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/lartpang/ZoomNeXt.git {REPO_DIR}
else:
    print(f'Repository already exists at {REPO_DIR}')

%cd {REPO_DIR}
print('Repository files:', sorted(os.listdir('.')))

# NOTE: we do NOT run `pip install -r requirements.txt` here.
# requirements.txt pins old versions (torch==2.1.2, opencv==4.8.1.78) that don't have
# Python 3.12 wheels. We only need the 6 packages from Cell 1 for inference.
# If you need full evaluation (metrics), see the optional cell at the bottom.

## Cell 3 — Set Up Paths

In [ ]:
# === Cell 3: Set Up Paths ===
import os, sys

REPO_DIR    = '/content/ZoomNeXt'
WEIGHT_DIR  = '/content/weights'
INPUT_DIR   = '/content/images/input'
OUTPUT_DIR  = '/content/images/output'
WEIGHT_FILE = 'resnet50-zoomnext.pth'
MODEL_NAME  = 'RN50_ZoomNeXt'

os.makedirs(WEIGHT_DIR, exist_ok=True)
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'DEVICE: {DEVICE}')
print(f'REPO_DIR: {REPO_DIR}')
print(f'WEIGHT_DIR: {WEIGHT_DIR}')

## Cell 4 — Download Weight

In [ ]:
# === Cell 4: Download Weight ===
weight_path = os.path.join(WEIGHT_DIR, WEIGHT_FILE)
FILE_ID = '1eN2EtzCKAlqLrN1NlkDi_v17dVnymLoW'

if os.path.exists(weight_path) and os.path.getsize(weight_path) > 1e8:
    print(f'Weight exists: {weight_path} ({os.path.getsize(weight_path)/1e6:.2f} MB)')
else:
    print(f'Downloading {WEIGHT_FILE}...')
    !gdown {FILE_ID} -O {weight_path}
    assert os.path.getsize(weight_path) > 1e8, 'Download failed'
    print(f'Downloaded: {os.path.getsize(weight_path)/1e6:.2f} MB')

## Cell 5 — Load Model + Verify

In [ ]:
# === Cell 5: Load Model + Verify ===
import time
import methods as model_zoo
from utils.io import load_weight

t0 = time.perf_counter()
model = model_zoo.__dict__[MODEL_NAME](num_frames=1, pretrained=False, use_checkpoint=False)
t1 = time.perf_counter()
n_params = sum(v.numel() for v in model.parameters())
print(f'Build: {t1-t0:.3f}s | Params: {n_params:,} ({n_params/1e6:.3f}M)')
assert n_params == 28458084, f'Param mismatch! Expected 28458084, got {n_params}'

t0 = time.perf_counter()
load_weight(model=model, load_path=weight_path, strict=True)
t1 = time.perf_counter()
print(f'Weight load: {t1-t0:.3f}s | strict=True, all keys matched')

model = model.to(DEVICE).eval()
print(f'Model ready on {DEVICE}')

## Cell 6 — Upload Image

In [ ]:
# === Cell 6: Upload Image ===
from google.colab import files
uploaded = files.upload()
IMAGE_NAME = list(uploaded.keys())[0]
IMAGE_PATH = os.path.join(INPUT_DIR, IMAGE_NAME)
with open(IMAGE_PATH, 'wb') as f:
    f.write(uploaded[IMAGE_NAME])
print(f'Image: {IMAGE_NAME} ({os.path.getsize(IMAGE_PATH)/1024:.1f} KB)')

## Cell 7 — Run Inference

In [ ]:
# === Cell 7: Run Inference ===
import cv2, numpy as np
from utils.ops import ms_resize, resize

image = cv2.imread(IMAGE_PATH)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
orig_h, orig_w = image_rgb.shape[:2]

images = ms_resize(image_rgb, scales=(0.5, 1.0, 1.5), base_h=384, base_w=384)
image_s = torch.from_numpy(images[0]).div(255).permute(2,0,1).unsqueeze(0).to(DEVICE)
image_m = torch.from_numpy(images[1]).div(255).permute(2,0,1).unsqueeze(0).to(DEVICE)
image_l = torch.from_numpy(images[2]).div(255).permute(2,0,1).unsqueeze(0).to(DEVICE)

t0 = time.perf_counter()
with torch.no_grad():
    logits = model(data={'image_s': image_s, 'image_m': image_m, 'image_l': image_l})
t1 = time.perf_counter()
INFERENCE_TIME = t1 - t0

probs = logits.sigmoid().squeeze().cpu().numpy()
probs = (probs - probs.min()) / (probs.max() + 1e-8)
pred = resize(probs, height=orig_h, width=orig_w)
pred_uint8 = (pred * 255).astype(np.uint8)
IMAGE_STEM = os.path.splitext(IMAGE_NAME)[0]
print(f'Inference: {INFERENCE_TIME:.3f}s | range [{pred.min():.4f}, {pred.max():.4f}] | mean {pred.mean():.4f}')

## Cell 8 — Save Output

In [ ]:
# === Cell 8: Save Output ===
pred_path = os.path.join(OUTPUT_DIR, f'{IMAGE_STEM}_pred.png')
cv2.imwrite(pred_path, pred_uint8)

input_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
pred_color = cv2.applyColorMap(pred_uint8, cv2.COLORMAP_JET)
vis = np.hstack([input_bgr, pred_color])
vis_path = os.path.join(OUTPUT_DIR, f'{IMAGE_STEM}_vis.png')
cv2.imwrite(vis_path, vis)

print(f'Saved: {pred_path}')
print(f'Saved: {vis_path}')

## Cell 9 — Visualization

In [ ]:
# === Cell 9: Visualization ===
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(image_rgb); axes[0].set_title(f'Input\n{IMAGE_NAME}'); axes[0].axis('off')
axes[1].imshow(pred, cmap='gray'); axes[1].set_title(f'Prediction\n[{pred.min():.3f}, {pred.max():.3f}]'); axes[1].axis('off')
axes[2].imshow(image_rgb); axes[2].imshow(pred, cmap='jet', alpha=0.5)
axes[2].set_title(f'Overlay\n{INFERENCE_TIME:.3f}s on {DEVICE}'); axes[2].axis('off')
plt.suptitle(f'RN50_ZoomNeXt | {n_params/1e6:.3f}M params | {IMAGE_NAME}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'{IMAGE_STEM}_result.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'\n=== DONE ===\nDevice: {DEVICE} | Time: {INFERENCE_TIME:.3f}s | Params: {n_params/1e6:.3f}M')

---
## (Optional) Full Evaluation Dependencies

Only run this if you need to compute metrics (Sm, Fw, MAE, Em) on a test dataset with ground truth.
This installs the additional packages needed by `main_for_image.py --evaluate`.

**WARNING:** This cell installs `albumentations` which pulls `numpy>=2`.
After running this cell, you MUST re-run `!pip install "numpy<2"` to fix torch compatibility.
Alternatively, use our standalone `infer.py` / `infer_tta.py` which do NOT need albumentations.

In [ ]:
# === Optional: Full Evaluation Dependencies ===
# These are needed ONLY for `python main_for_image.py --evaluate` (metrics computation).
# NOT needed for single-image inference (Cells 1-9).
#
# KEY INSIGHT: Modern torch (2.11+) supports numpy 2.x. So we install the LATEST
# scientific stack (compiled against numpy 2.x). DO NOT pin old versions like
# scipy==1.13.1 or scikit-image==0.22.0 — those were compiled against numpy 1.x
# and will crash with "numpy.dtype size changed" on numpy 2.x.

# Install latest scientific stack (all numpy-2 compatible).
!pip install -q scipy scikit-image scikit-learn mmengine PyYAML matplotlib colorlog tqdm

# pysodmetrics with --no-deps: it requires opencv-python-headless<5.0.0, but Colab
# ships opencv 5.x. The actual code works fine with opencv 5.x, so --no-deps avoids
# the version conflict.
!pip install -q --no-deps pysodmetrics

# Verify evaluation dependencies
import numpy, mmengine, yaml, scipy, matplotlib, colorlog, tqdm
import py_sod_metrics
print('Full evaluation dependencies installed.')
print(f'numpy={numpy.__version__}')
print(f'scipy={scipy.__version__}')
print(f'pysodmetrics: OK')
print(f'mmengine: {mmengine.__version__}')
print()
print('Note: albumentations is NOT installed (it breaks torch via numpy>=2 on some systems).')
print('For inference without metrics, use infer.py / infer_tta.py instead.')